# Rico local-model benchmark — Colab / Kaggle GPU runner

Runs **Ollama + GGUF** on a free cloud GPU, using the **same `bench_ollama.py`**
harness as the PC. Deliberately *not* HuggingFace Transformers + FP16:
Rico would deploy Ollama + Q4_K_M, and quantization changes both quality and
refusal boundaries — so an FP16 run measures the model's ceiling, not what you
would actually get. Same runtime + same quant = results transfer to the PC.

## What this answers, and what it does NOT

| Transfers to your PC | Does NOT transfer |
|---|---|
| Arabic / English quality | tok/s (prefill, decode) |
| JSON reliability (20 attempts) | first-token latency |
| Over-refusal on HR tasks | VRAM fit on 6 GB |
| Reasoning, coding | CPU/GPU split |
| CV ↔ JD matching | whether CUDA works on the GTX 1060 |

**Speed numbers here are about the cloud GPU, not your GTX 1060. Never quote
them as your hardware's performance.** The harness stamps every row with
`run_mode` so cloud and PC results cannot be silently mixed.

## Scope

Evaluation only. Touches no Rico production code, config, DB, deployment or
AI routing. All benchmark inputs are **synthetic** — no real CV, no PII.

## Platform notes

- **Kaggle is usually the better choice**: ~29 GB system RAM and a documented
  ~30 GPU-hours/week, vs Colab Free's ~12.7 GB RAM and undocumented dynamic limits.
- **Counter-intuitive**: `Qwen3.6-35B-A3B` (~18–20 GB) does **not** fit a 16 GB
  T4 and would spill to system RAM — and Colab Free has *less* RAM than your
  32 GB PC. **That model belongs on your PC, not on Colab Free.**
- Sessions are ephemeral: every restart re-downloads models. Work in **batches
  of 2–3 models**, don't attempt all seven in one session.

## 1. What GPU did we actually get?

Free tiers assign GPUs by availability. Record what you got — it is part of
the result metadata. **If this cell shows no GPU, stop**: change the runtime
type to GPU, or the whole session runs on a weak vCPU.

In [ ]:
import subprocess, sys

try:
    out = subprocess.run(
        ["nvidia-smi",
         "--query-gpu=name,driver_version,compute_cap,memory.total",
         "--format=csv"],
        capture_output=True, text=True, check=True,
    ).stdout
    print(out)
    HAS_GPU = True
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    print(f"NO GPU DETECTED ({exc}). Runtime -> Change runtime type -> GPU.")
    HAS_GPU = False

print("--- system RAM / disk ---")
print(subprocess.run(["free", "-g"], capture_output=True, text=True).stdout)
print(subprocess.run(["df", "-h", "/"], capture_output=True, text=True).stdout)

## 2. Install Ollama

Ollama is an ordinary Linux binary, so it installs and runs fine in a notebook
VM. This is what keeps the runtime identical to the PC setup.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

## 3. Start the Ollama server in the background

Then wait until the API actually answers — do not assume it is up.

In [ ]:
import os, subprocess, time, urllib.request, urllib.error

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("ollama_server.log", "w"),
    stderr=subprocess.STDOUT,
)

def wait_for_api(timeout=90):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=3):
                return True
        except (urllib.error.URLError, OSError):
            time.sleep(2)
    return False

print("Ollama API is up" if wait_for_api() else "API did not come up — see ollama_server.log")

## 4. Choose the models for THIS session

Two or three per session. Downloads are the real cost, not GPU time.

**Verify every tag before running.** Tags in `MODEL_DISCOVERY.md` are marked
`[UNVERIFIED]` because `ollama.com` and `huggingface.co` were blocked from the
environment that wrote them. A wrong tag wastes the session.

Specifically unconfirmed:
- **Gemma-4 E4B uncensored** — Gemma 4 exists (Apache 2.0); a specific
  uncensored E4B build was never confirmed. If it does not exist, substitute
  base Gemma 4 E4B and record it as an aligned (Category C) contrast.
- **Falcon-H1** is a hybrid Mamba-Transformer. Support may lag. The next cell
  load-tests each model *before* the benchmark for exactly this reason.

In [ ]:
# Batch 1 — smallest first, so a broken toolchain costs minutes, not 40 GB.
MODELS = [
    "huihui_ai/qwen3-abliterated:4b",
    "qwen2.5:7b",  # control — anchors cloud results to existing CPU evidence
]

# Batch 2 (later session):
#   "huihui_ai/qwen3.5-abliterated:<verify-tag>"
#   "hf.co/tiiuae/Falcon-H1-Arabic-7B-Instruct-GGUF"
# Do NOT put Qwen3.6-35B-A3B here on Colab Free — ~18-20 GB exceeds a 16 GB T4
# and Colab Free's ~12.7 GB RAM is less than your PC's 32 GB. Run it on the PC.

for tag in MODELS:
    print(f"=== pulling {tag} ===", flush=True)
    subprocess.run(["ollama", "pull", tag], check=False)

print("\n=== local models ===")
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

## 5. Load test + prove the GPU is really being used

A silent CPU fallback that gets benchmarked as "GPU" is the worst outcome
here, so check before spending the session. `ollama ps` shows the CPU/GPU
split directly.

In [ ]:
import json, urllib.request

def vram_used_mib():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True,
        ).stdout.strip().splitlines()[0]
        return int(out)
    except Exception:
        return -1

for tag in MODELS:
    print(f"\n=== {tag} ===")
    before = vram_used_mib()

    payload = json.dumps({
        "model": tag,
        "prompt": "Reply with the single word: OK",
        "stream": False,
        "options": {"num_gpu": 99, "temperature": 0, "num_ctx": 2048},
    }).encode()
    req = urllib.request.Request(
        "http://127.0.0.1:11434/api/generate", data=payload,
        headers={"Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(req, timeout=300) as r:
            body = json.loads(r.read().decode())
        print("  loaded OK, response:", repr(body.get("response", ""))[:80])
    except Exception as exc:
        print(f"  LOAD FAILED: {exc}  -> drop this model, do not debug it now")
        continue

    after = vram_used_mib()
    print(f"  VRAM {before} -> {after} MiB (delta {after - before})")
    print("  ollama ps:")
    print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout)

print("\nConfirm before continuing: PROCESSOR reads '100% GPU' and the VRAM delta")
print("is roughly the model size. A ~0 delta means it is running on CPU.")

## 6. Get the harness

Upload `bench_ollama.py` from `AI_WORKSPACE/EVALS/local-model-eval/`.
Upload is used rather than `git clone` because the repo is private — cloning
would mean putting a token into a notebook on a shared VM, which is not worth
it for one file.

In [ ]:
import os

if not os.path.exists("bench_ollama.py"):
    try:
        from google.colab import files  # type: ignore
        files.upload()  # select bench_ollama.py
    except ImportError:
        print("Not on Colab. On Kaggle: add the file via 'Add Data' / the file panel.")

assert os.path.exists("bench_ollama.py"), "bench_ollama.py not found"
!python bench_ollama.py --list-suite

## 7. Run the benchmark

`--num-gpu 99` is **mandatory** here. The harness defaults to `num_gpu=0`
(forced CPU) because the PC's GPU is broken — leaving the default on a GPU
runner silently spends the whole session on a weak vCPU.

In [ ]:
!python bench_ollama.py \
  --models {" ".join(MODELS)} \
  --num-gpu 99 \
  --timeout 600 \
  --out results_cloud

## 8. Save the results before the session dies

Sessions are ephemeral and can be reclaimed without warning. Download
immediately — an unsaved benchmark is a wasted session.

In [ ]:
import shutil, datetime

stamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
archive = f"rico_bench_cloud_{stamp}"
shutil.make_archive(archive, "zip", "results_cloud")
print(f"{archive}.zip")

try:
    from google.colab import files  # type: ignore
    files.download(f"{archive}.zip")
except ImportError:
    print("On Kaggle: the zip is in the working dir; save it via Output.")

## 9. After the session

Commit the zip to `AI_WORKSPACE/EVALS/local-model-eval/evidence/` — it is
synthetic-input evidence with no secrets and no PII, and single-copy evidence
on one machine is finding **F-0** in `REVIEW.md`.

**Record alongside the results:** which GPU the session assigned, the Ollama
version, and each model's digest + quantization (the harness captures the last
two automatically).

### What this does and does not settle

- **Settled:** which models are good enough on quality, Arabic, JSON
  reliability and refusal behaviour → narrows 7 candidates to 2–3.
- **Not settled:** whether any of them is *fast enough on a GTX 1060*, and
  whether that GPU works at all. That stays Stage 0–3 of
  `BENCHMARK_EXECUTION_PLAN.md`.

Doing the cloud pass first is strictly better sequencing: you learn which
model you want *before* investing a session in GPU recovery, and if recovery
fails you can decide about hardware with the quality question already answered.